<style>
/* Limit text outputs for all cells */
div.output_area pre {
    max-height: 300px;
    overflow: auto;
}
/* Limit rich outputs */
div.output_area div.output_subarea {
    max-height: 300px;
    overflow: auto;
}
</style>

# Deep PheWAS Step 1
## 0. Set the working directory
**ALWAYS DO THIS AT THE START OF THE SESSION BEFORE RUNNING ANY CELLS BELOW**


In [ ]:
cd /opt/notebooks/deep_phewas_RAP_install

If you want to close the jupyter session and resume the workflow later remember to `Create Snapshot` from the `DNAnexus` menu and then start your new jupyter lab session specifying the saved snapshot to load. 

Snapshots are saved in the root of your project under `/.Notebook_snapshots` with the date in the filename.
## 1. Set up the environment
###    a) Install Java
Java is required for the WDL compiler.

In [ ]:
apt update -qq
apt install -y -qq openjdk-8-jre

### b) Install R tidyverse
The Deep PheWAS R install scripts use tidyverse.

In [ ]:
Rscript -e 'install.packages("tidyverse", quiet=TRUE)'

## 2. Install the WDL workflows to your RAP project

### a) Set the RAP directory where you will install Deep PheWAS
Edit the `options.config` file and set `PROJECT_DIR` e.g.
```bash
PROJECT_DIR=/deep_phewas
```

### b) Run the install_workflows.sh script
This script will:
* Create a docker image in your project containing the DeepPheWAS package, plink2 and dependencies.
* Download DNANexus dxCompiler required to compile WDL workflows to DNANexus workflows to run on the RAP.
* Compile and install the WDL DeepPheWAS workflows in your RAP project.

In [ ]:
./install_workflows.sh

## 3. Extracting the UK Biobank data fields that provide the input for phenotype generation
### a) Find the most up to date UK Biobank phenotype data

In [ ]:
dx ls -l /*.dataset

Edit options.config and set the data set to the latest version above e.g. `DATASET=/app648_20250722114356.dataset`
### b) Extract the phenotype data from the most up-to-date source
Run `./extract_fields.sh`, which will launch several RAP jobs to extract:
* Participant data fields
* Hospital episode statistics (HES)
* Death registry data
* Primary care data

Wait for these jobs to finish.


In [ ]:
./extract_fields.sh

## 4. Phenotype generation
### a) Make the configuration file specifying the inputs for phenotype generation
Run `make_inputs_phenotype_generation.sh` which will make the .json configuration file specifying the inputs required for the phenotype generation step.

**Ignore warnings** `missing input for non-optional parameter`

`Intermediate representation` means it has run correctly.

In [ ]:
./make_inputs_phenotype_generation.sh

### b) Run the phenotype generation
Run `run_phenotype_generation.sh` which will submit a RAP analysis pipeline to generate the phenotypes.

The pipeline submits several sub-jobs that run in parallel to make:
* Data field phenotypes
* Phecode phenotypes
* Primary care phenotypes
* Formula phenotypes
* Composite phenotypes

Wait until the pipeline has completed before proceeding.

In [ ]:
./run_phenotype_generation.sh

## 5. Phenotype preparation
### a) Make the configuration file specifying the inputs for phenotype preparation
In `options.config` you can change:
* Whether to remove related samples (related_remove=true/false, default=false)
* Whether to rank inverse-normal transform phenotypes (IVNT=true/false, default=true)
* Any groupings of samples, e.g. by ancestry (groupings=ancestry_panUKB)

Run `make_inputs_phenotype_preparation.sh` which will make the .json configuration file specifying the inputs required for the phenotype preparation step.

`Intermediate representation` means it has run correctly.

In [ ]:
./make_inputs_phenotype_preparation.sh

### b) Run the phenotype preparation
Run `run_phenotype_preparation.sh` which will submit the RAP job to create the phenotype tables to be used in association testing.

Wait until the pipeline has completed before proceeding.

In [ ]:
./run_phenotype_preparation.sh

## 6. Regenie Step 1
### a) Extract the covariates to use in the association testing from the UK Biobank data
In `options.config` you can change:
* The name of the covariate file to create (covar=, default=covar_regenie.txt)
* The names of the continuous covariate columns to include (covarColList=, default=age,PC{1:10})
* The names of categorical covariates to include (catCovarList=, defaults=sex,array)

Run `extract_covar.sh` and wait for the RAP job to finish.

In [ ]:
./extract_covar.sh

### b) Make the configuration file specifying the inputs for regenie step 1
In `options.config` you can set:
* The IDs of phenotypes to include if you only want to use a subset of all phenotypes available (phenoColList=).

Run `make_inputs_regenie_step1.sh` which will make the .json configuration file specifying the inputs required for regenie step 1.

**Ignore warnings** `missing input for non-optional parameter`

`Intermediate representation` means it has run correctly.

In [ ]:
./make_inputs_regenie_step1.sh

### Run regenie step 1
Run `run_regenie_step1.sh` which will submit a RAP workflow that runs the following tasks:
* Divides the phenotype files from the Phenotype preparation step above into binary and quantitative traits
* Clusters the traits by patterns of missingness such that no cluster has any trait with >15% missingness (value can be altered in `RAP.config`).
* For each cluster of traits a set of SNPs is filtered to have --geno 0.1 --hwe 1e-15 --mac 100 --maf 0.01 --mind 0.1

Regenie step 1 is run on each cluster.

In [ ]:
./run_regenie_step1.sh

The output will be in ${PROJECT_DIR}/step1 and comprises:
* A set of _pred.list files 1 for each sample grouping as specified by the grouping file (usually grouped for ancestry), combined across all phenotypes.
* A set of .loco files, 1 for each sample grouping and phenotype.